In [3]:
!pip install sentence-transformers chromadb groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [4]:
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported")

All libraries imported


In [5]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("API key Initialised")

API key Initialised


In [6]:
df = pd.read_csv('/content/drive/MyDrive/Data_Engineering_Internship/college_notes.csv')
print(f"Column Name : {df.columns.tolist()}")
print(f"Shape :  {df.shape}")
print(df.head(3))
print(df.tail(3))

Column Name : ['note_id', 'subject', 'topic', 'content']
Shape :  (15, 4)
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
   note_id             subject                           topic  \
12    N013       Generative AI  Retrieval Augmented Generation   
13    N014  Python Programming                  Pandas Library   
14    N015  Python Programming              Data Visualization   

                                              content  
12  RAG or Retrieval Augmented Generation is a tec...  
13  Pandas is a Python library used for data manip...  
14  Data visualization is the process of represent...  


In [7]:
print("Subjects in the dataset")
print(df['subject'].value_counts)

print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content of each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subjects in the dataset
<bound method IndexOpsMixin.value_counts of 0       Data Engineering
1       Data Engineering
2       Data Engineering
3       Data Engineering
4       Data Engineering
5       Machine Learning
6       Machine Learning
7       Machine Learning
8       Machine Learning
9       Machine Learning
10         Generative AI
11         Generative AI
12         Generative AI
13    Python Programming
14    Python Programming
Name: subject, dtype: object>

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Featur

In [8]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared : {len(documents)}")
print(f"First document ID     : {ids[0]}")
print(f"First document Meta   : {metadatas[0]}")
print(f"First 100 char of docc: {documents[0][:100]}")
print()
print(f"14 Document IDs       : {ids[:14]}")
print(f"14 Document meta      : {metadatas[:14]}")
print(f"First 100 char of docc: {documents[14][:100]}")

Total chunks prepared : 15
First document ID     : note_N001
First document Meta   : {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 char of docc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc

14 Document IDs       : ['note_N001', 'note_N002', 'note_N003', 'note_N004', 'note_N005', 'note_N006', 'note_N007', 'note_N008', 'note_N009', 'note_N010', 'note_N011', 'note_N012', 'note_N013', 'note_N014']
14 Document meta      : [{'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}, {'subject': 'Data Engineering', 'topic': 'SQL Databases'}, {'subject': 'Data Engineering', 'topic': 'Data Cleaning'}, {'subject': 'Data Engineering', 'topic': 'APIs and Data Collection'}, {'subject': 'Data Engineering', 'topic': 'Big Data and PySpark'}, {'subject': 'Machine Learning', 'topic': 'Supervised Learning'}, {'subject': 'Machine Learning', 'topic': 'Model Evaluation'}, {'subject': 'Machine Learning', 'topic': 'Feature Engineeri

In [9]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
test_embedding = embedding_model.encode("This is a test sentence")

print(f"Test embedding shape: {test_embedding.shape}")
print(f"first 10 values of test embedding: {test_embedding[:5]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Test embedding shape: (384,)
first 10 values of test embedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [10]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")

print("Chromadb client created")
print(f"Collection name : college_notes_rag")
print(f"Document in collection : {collection.count()}")

Chromadb client created
Collection name : college_notes_rag
Document in collection : 0


In [11]:
print("Generating embeddings....")

embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"Embeddings shape : {embeddings.shape}")
embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)

print(f"Document in collection : {collection.count()}")


Generating embeddings....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape : (15, 384)
Document in collection : 15


In [12]:
def retrieve_relevant_chunks(question, top_k = 3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings = [question_embedding],
      n_results=top_k
  )
  return results


In [13]:
test_question = "What is GenAI and how does it work in Embedding?"
print(f"Test question :{test_question}")
print("="*60)

results = retrieve_relevant_chunks(test_question,top_k=3)
print("Top 3 Retireved Chunks:")
print("="*60)
print()

for i,(doc,dist,meta) in enumerate(zip(results['documents'][0],results['distances'][0],results['metadatas'][0])):
  print(f"Results{i+1}:")
  print(f"Subject : {meta['subject']}")
  print(f"Topic : {meta['topic']}")
  print(f"Distance : {dist:.4f}")
  print(f"Content : {doc[:120]}...")
  print()

Test question :What is GenAI and how does it work in Embedding?
Top 3 Retireved Chunks:

Results1:
Subject : Generative AI
Topic : Retrieval Augmented Generation
Distance : 1.1898
Content : RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowled...

Results2:
Subject : Generative AI
Topic : Large Language Models
Distance : 1.3646
Content : A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-like text an...

Results3:
Subject : Machine Learning
Topic : Feature Engineering
Distance : 1.5740
Content : Feature engineering is the process of selecting transforming and creating input variables that help machine learning mod...



In [21]:
def build_context_from_results(results):
  context_parts = []

  for i,(doc,meta) in enumerate(zip(results['documents'][0],results['metadatas'][0])):
    chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)

  context_str = "\n\n--\n\n".join(context_parts)

  return context_str

context = build_context_from_results(results)
print(context[:500]+"...")
print("Total context length ",len(context))

[Source 1: Generative AI - Retrieval Augmented Generation]
RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.

--

[Source 2: Generative AI - Large Language Models]
A Large Language Model or LLM is an AI model trained on massive amounts of text data. It can generate human-lik...
Total context length  908


In [22]:
def generate_rag_answer(question,context):
  system_prompt = """
    You are a helpful academic assistant for engineering students.


    You will be given context retrieved from a college knowledge base, and a student's question.


    RULES:
    1. Answer ONLY using the information provided in the context below.
    2. If the answer is not found in the context, say exactly:
       "I don't have enough information in my knowledge base to answer this question."
    3. Do not use your general training knowledge.
    4. Keep answers clear, accurate, and beginner-friendly.
    5. Mention which source the information came from when possible."""
  user_prompt = f"""Context from Knowledge Base:{context}
  ---
  Student's Question: {question}
  Please answer the question based only on the context provided above."""

  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
    )
  answer = response.choices[0].message.content
  return answer
print("RAG CREATED SUCCESFULLY")

RAG CREATED SUCCESFULLY


In [23]:
def ask_college_assistant(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question  : {question}")
    print("="*60)

  results = retrieve_relevant_chunks(question, top_k=top_k)

  if verbose:
    print(f"Restricted {top_k} chunks from the knowledge base:")
    for i, meta in enumerate(results['metadatas'][0]):
      print(f"  {i+1}. {meta['subject']} - {meta['topic']}")

    print("\n Building context string")

  context = build_context_from_results(results)

  if verbose:
    print(f"Context built({len(context)} characters)")
    print("\n Sending to llm for answer generation")

  answer = generate_rag_answer(question, context)

  if verbose:
    print("\n"+"="*60)
    print("\nAnswer Generated:")
    print("="*60)
    print(answer)
    print("="*60)

  return answer

print("Completed RAG")

Completed RAG


In [24]:
question1 = "What is ETL and what are its three main stages"
answer1 = ask_college_assistant(question1, top_k=3, verbose=True)

Question  : What is ETL and what are its three main stages
Restricted 3 chunks from the knowledge base:
  1. Data Engineering - ETL Pipelines
  2. Generative AI - Retrieval Augmented Generation
  3. Generative AI - Prompt Engineering

 Building context string
Context built(928 characters)

 Sending to llm for answer generation


Answer Generated:
Based on the context provided, I can answer the student's question.

ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis. [Source 1: Data Engineering - ETL Pipelines]

The three main stages of ETL are:

1. **Extract**: Collecting raw data from different sources.
2. **Transform**: Transforming the raw data into a clean and structured format.
3. **Load**: Loading the transformed data into a database or data warehouse for analysis.

These stages are mentioned in the context of ETL Pi

In [26]:
question2 = "what is machine learning"
answer2 = ask_college_assistant(question2, top_k=3, verbose=True)

Question  : what is machine learning
Restricted 3 chunks from the knowledge base:
  1. Machine Learning - Supervised Learning
  2. Machine Learning - Decision Trees
  3. Machine Learning - Random Forest

 Building context string
Context built(874 characters)

 Sending to llm for answer generation


Answer Generated:
Based on the context provided, I can answer the student's question.

Machine learning is not explicitly defined in the given context. However, we can infer that it is a type of learning where a model learns from data. 

From [Source 1: Machine Learning - Supervised Learning], we know that supervised learning is a type of machine learning where the model learns from labeled data. This suggests that machine learning involves learning from data.

However, I don't have enough information in my knowledge base to provide a comprehensive definition of machine learning based on the given context.


In [27]:
question3 = "Who is the prime minister of india"
answer3 = ask_college_assistant(question3, top_k=3, verbose=True)


Question  : Who is the prime minister of india
Restricted 3 chunks from the knowledge base:
  1. Generative AI - Large Language Models
  2. Machine Learning - Supervised Learning
  3. Machine Learning - Random Forest

 Building context string
Context built(877 characters)

 Sending to llm for answer generation


Answer Generated:
I don't have enough information in my knowledge base to answer this question.

The context provided only includes information about Large Language Models, Supervised Learning, and Random Forest, but does not contain any information about the current or past prime ministers of India.
